In [1]:
import sys
from pathlib import Path

In [2]:
PROJECT_ROOT = Path().resolve().parent.parent.parent

In [3]:
# Add project root to Python path
sys.path.append(str(PROJECT_ROOT))

In [4]:
import pandas as pd
from src.pipeline.utils.loader import load_jsonl
from src.pipeline.utils.document import Document
from collections import Counter
from typing import Any, Dict, Iterator
import plotly.express as px

In [5]:
DATA_PATH = PROJECT_ROOT / "outputs" / "pipeline_test" / "2_minhash_deduplication" / "keep"

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "processed"

In [ ]:
def extract_field_paths(obj: Any, prefix: str = "") -> set[str]:
    """
    Recursively extract all field paths from nested metadata.

    Example:
    {
        "author": {
            "name": "Alice"
        },
        "tags": [{"label": "x"}]
    }

    -> {
        "author",
        "author.name",
        "tags",
        "tags[].label"
    }
    """
    fields = set()

    if isinstance(obj, dict):
        for key, value in obj.items():
            path = f"{prefix}.{key}" if prefix else key

            fields.add(path)

            fields.update(extract_field_paths(value, path))

    elif isinstance(obj, list):
        for item in obj:
            list_prefix = f"{prefix}[]"
            fields.update(extract_field_paths(item, list_prefix))

    return fields


In [ ]:
def analyze_documents(doc_iter: Iterator[Document]):
    total_docs = 0
    field_counts = Counter()

    for doc in doc_iter:
        total_docs += 1

        fields_in_doc = extract_field_paths(doc.metadata)

        # Count each field once per document
        field_counts.update(fields_in_doc)

    return total_docs, field_counts

In [6]:
def iterate_directory(directory:Path) -> Iterator[Document]:
    for file in directory.glob('*.jsonl'):
        documents = load_jsonl(str(file), field_map={"id":"id", "text":"text"}, load_metadata=True, metadata_fields="*")
        for doc in documents:
            yield doc

In [7]:
documents = iterate_directory(DATA_PATH)

In [9]:
import json

In [11]:
affected_docs = ["americasnlp2024_27377", "americasnlp2024_27378", "fineweb-2_365", "fineweb-2_748", "fineweb-2_1651", "fineweb-2_3042",
                      "fineweb-2_4016"," fineweb-2_4500", "fineweb-2_7468", "fineweb-2_11951", "fineweb-2_12334", "fineweb-2_13237", "fineweb-2_14628",
                      "fineweb-2_15602", "fineweb-2_16086", "fineweb-2_19054", "fineweb-2_23537", "fineweb-2_23920", "fineweb-2_24823", "fineweb-2_26214",
                      "fineweb-2_27188", "fineweb-2_27672", "fineweb-2_30640", "fineweb-2_36209", "fineweb-2_37078", "fineweb-2_37116", "fineweb-2_37527",
                      "fineweb-2_38176", "fineweb-2_41411", "fineweb-2_43121", "fineweb-2_44199", "fineweb-2_44993"]

In [ ]:
#corr = df.corr(numeric_only=True)

In [ ]:
# fig = px.imshow(
#     corr,
#     text_auto=".2f",
#     color_continuous_scale="RdBu_r",
#     zmin=-1,
#     zmax=1,
#     aspect="auto",
# )

# fig.update_layout(
#     title="Correlation Heatmap",
#     width=1000,
#     height=1000,
# )

# fig.show()

In [ ]:
# fig = px.box(
#     df, 
#     x="remove",           # Primary category on X-axis
#     y="avg_alphanumeric characters_per_c",     # Quantitative values on Y-axis
#     color="remove",     # Separate sub-categories by color
#     title="Grouped Box Plot with Separate Categories"
# )

# # Display the plot
# fig.show()

In [ ]:
# fig = px.scatter(
#     df,
#     x="language_score",
#     y="avg_numbers_per_sentence",
#     #size="language_score",          # dot size
#     color="remove",              # dot colour
#     hover_data=["id"],
#     size_max=40,
#     opacity=0.7,
#     title="Low Guarani Proportion vs Javascript Sentences"
# )

# fig.show()

In [8]:
rows_all = []
for document in documents:
    row = {
        "id": document.id,
        "text": document.text
    }
    for field, value in document.metadata.items():
        if type(value) != "dict":
            row[field] = value

    #row["remove"] = document.id in affected_docs

    rows_all.append(row)

In [ ]:
df_all = pd.DataFrame(rows_all)

In [15]:
df_all.columns

Index(['id', 'text', 'num_words_no_punct_spacy', 'duplicate', 'num_chars',
       'corpus_file', 'avg_word_repetition_ratio_per_sentence',
       'avg_words_per_sentence', 'avg_sentence_length',
       'count_lorem_ipsum_sentences', 'avg_uppercase_letters_per_sentence',
       'count_sentences_with_low_guarani_proportion',
       'avg_characters_per_sentence', 'count_sentences_starting_with_bullet',
       'language_score_source',
       'average_words_in_sentences_starting_with_capital',
       'count_sentences_ending_with_ellipsis',
       'count_sentences_with_legal_phrases', 'url', 'language_score',
       'num_words_punct_spacy', 'ratio_symbols_to_words',
       'avg_alphanumeric_characters_per_sentence',
       'count_sentences_without_terminal_punctuation',
       'language_identification_method', 'num_words_split', 'language',
       'count_sentences_with_curly_bracket', 'source', 'language_script',
       'count_bad_words_occurrences', 'max_sentence_length',
       'count_sent

In [14]:
df_all["proportion_sentences_with_low_guarani_score"] = (df_all["count_sentences_with_low_guarani_proportion"])/(df_all["num_words_split"]/df_all["avg_words_per_sentence"])

In [25]:
df_all["language_coefficient"] = df_all['language_score']/df_all["proportion_sentences_with_low_guarani_score"]

In [15]:
df_all['remove_js'] = df_all['id'].isin(affected_docs)

In [ ]:
fig = px.scatter(
    df_all[df_all['ratio_symbols_to_words']>1.27],
    x="language_score",
    y="ratio_symbols_to_words",
    #size="language_score",          # dot size
    #color="remove",              # dot colour
    hover_data=["id"],
    size_max=40,
    opacity=0.7,
    title="Low Guarani Proportion vs Javascript Sentences"
)
fig.show()

In [ ]:
fig = px.box(
    df_all,
    x="proportion_sentences_with_low_guarani_score",
    points="outliers"
)

fig.show()

In [ ]:
df_all[df_all['num_words_punct_spacy']<2].to_json(str(PROJECT_ROOT/ 'outputs'/ 'heuristics_filtering' /'n_words_l_2.jsonl'), orient='records', lines=True, force_ascii=False)

In [12]:
df_all[df_all['language_score']<0.7].to_json(str(PROJECT_ROOT/ 'outputs'/ 'heuristics_filtering' /'low_lang_score.jsonl'), orient='records', lines=True, force_ascii=False)

In [13]:
df_all[df_all['language_score']==0.0].to_json(str(PROJECT_ROOT/ 'outputs'/ 'heuristics_filtering' /'low_lang_score_0.jsonl'), orient='records', lines=True, force_ascii=False)

In [15]:
df_all[df_all['language_score']==1.0][['text', 'language_score', 'language_identification_method']]

,text,language_score,language_identification_method
13772,adura jave heta acto ñemotenonde “oîva permiti...,1.0,agree_glotlib_fasttext_openlid
14114,Ára 31 jasypakõi jave mayma yvypóra oñembosako...,1.0,agree_glotlib_fasttext_openlid
25489,"1995: Francisco Grande Covián, bioquímico Epáñ...",1.0,agree_glotlib_fasttext_openlid
26392,Delaware ha'e niko peteĩva Tetãvore Joapykuéra...,1.0,agree_glotlib_fasttext_openlid
26658,Vatõpuku () pa’i ao oipurúva ñembo’e guasurã.,1.0,agree_glotlib_fasttext_openlid
...,...,...,...
936078,Emombe’u peteĩ evento social nda’aréi reho ha...,1.0,agree_glotlib_fasttext_openlid
944907,Embojoaju mokõive temiandu eipurúvo ñe’ẽ tran...,1.0,agree_glotlib_fasttext_openlid
945946,Eme’ẽ peteĩ ñe’ẽ alternativo pe ñe’ẽ oñeme’ẽv...,1.0,agree_glotlib_fasttext_openlid
953480,Ehai jey ko ñe’ẽjoaju reiporúvo peteĩ ñe’ẽ pe...,1.0,agree_glotlib_fasttext_openlid


In [ ]:
fig = px.box(
    df_all, 
    #x="remove",           # Primary category on X-axis
    y="num_words_spacy",     # Quantitative values on Y-axis
    #color="remove",     # Separate sub-categories by color
    #title="Grouped Box Plot with Separate Categories"
)

# Display the plot
fig.show()

In [70]:
df_all[df_all['remove_js'] == True][['text', 'proportion_sentences_with_low_guarani_score', 'language_coefficient', 'count_sentences_with_low_guarani_proportion', 'language_score']]

,text,proportion_sentences_with_low_guarani_score,language_coefficient,count_sentences_with_low_guarani_proportion,language_score
505939,[JavaScript rembipuru'i],0.250000,3.332241,1,0.833060
839144,Cercanos\nIr a la navegación\nIr a la búsqueda...,0.424242,2.267862,2,0.962123
839511,Contribuciones del usuario Wikitanvir\nIr a la...,0.000000,inf,0,0.939448
840395,Contribuciones del usuario Wikitanvir\nIr a la...,0.000000,inf,0,0.844276
841763,Oñemoambue pyahúva\nSigue los cambios más reci...,0.274943,2.542887,65,0.699148
842709,Ayuda\nZona de pruebas de la API\nIr a la nave...,0.299065,3.315320,1,0.991498
846094,"Cercanos\nKundaharãme jeho\nkundaharã\n,\nJehe...",0.391858,2.541678,2,0.995976
850086,Firefox Browser Developer Edition\nEg̃uahẽporã...,0.182140,5.484668,7,0.998978
850645,BETA\nDisplaying 1 result:\n|Entity||en-US||gn...,0.361486,2.619921,3,0.947066
850667,Displaying 1 result:\n|Entity||en-US||gn|\n|En...,0.444444,2.121960,4,0.943094


In [77]:
df_all[(df_all['remove_js']==False) & (df_all['proportion_sentences_with_low_guarani_score']>2)][['text','proportion_sentences_with_low_guarani_score', 'count_sentences_with_low_guarani_proportion']]

,text,proportion_sentences_with_low_guarani_score,count_sentences_with_low_guarani_proportion
496006,ñoñe’ẽme’ẽ,3.0,1
496012,mba’evai’apo,3.0,1
496049,jepy’apy’ỹ,3.0,1
496096,ñe’ẽme’ẽ,3.0,1
498628,hepyme’ẽjo’a,3.0,1
499674,mba’ejoja’ỹ,3.0,1
500990,Ñe’ẽmondo’ỹ,3.0,1
501040,Mba’e’oka oku’éva,2.5,1
501931,Ñe’ẽasaha mba’apoha’aty,2.5,1
502036,Ha’arõ’ỹva,3.0,1


In [20]:
df_all[(df_all['proportion_sentences_with_low_guarani_score']>0.8)][['text','proportion_sentences_with_low_guarani_score', 'count_sentences_with_low_guarani_proportion', 'language_score']]

,text,proportion_sentences_with_low_guarani_score,count_sentences_with_low_guarani_proportion,language_score
5,Maitei peeme guarã,1.000000,1,0.851744
7,Napende kuerai piko la esquema preferidogui🙄,1.000000,1,0.860809
9,Ipochyma @SalvadorHicar “para eso nomás piko”,0.833333,1,0.644967
13,Chembohasy la arriero kuera😑,1.000000,1,0.970571
22,Nde tavy anocheeeeeeee🥴🥴🥴,1.000000,1,0.000000
...,...,...,...,...
960105,Pe entrenamiento de resistencia ombohetave pe ...,0.861538,4,0.999738
960123,Pe concentración glucosa tuguýpe 80-90 minuto ...,0.828571,3,0.994692
960124,Peteĩ mba’e ojehechavéva ojejapo haguã estoma ...,0.866667,4,1.000005
960133,Mba e hormonapa oguereko tenonderãite pe regul...,0.833333,5,0.999428


In [91]:
df_all[df_all['language_score'] == 0.0]

,id,text,num_words_no_punct_spacy,duplicate,num_chars,corpus_file,avg_word_repetition_ratio_per_sentence,avg_words_per_sentence,avg_sentence_length,count_lorem_ipsum_sentences,...,count_sentences_with_javascript,avg_numbers_per_sentence,corpus,ratio_stopwords_to_non_stopwords,mean_word_length,avg_character_repetition_ratio_per_sentence,min_sentence_length,proportion_sentences_with_low_guarani_score,language_coefficient,remove_js
4,joff+_4,@Pollo2895 Ko.agaite peve,3,"{'url': {'has_duplicate': False, 'docs_ids': []}}",25,main_off_lang_dev.txt,0.000000,1.0,1.0,0,...,0,4.0,joff+,0.000000,4.000000,0.260870,1,0.333333,0.0,False
10,joff+_10,reko ☹️☹️☹️,7,"{'url': {'has_duplicate': False, 'docs_ids': []}}",11,main_off_lang_dev.txt,0.000000,1.0,1.0,0,...,0,0.0,joff+,0.000000,4.000000,0.400000,1,0.142857,0.0,False
14,joff+_14,@renjunrohayhu Y si mante 😫,5,"{'url': {'has_duplicate': False, 'docs_ids': []}}",27,main_off_lang_dev.txt,0.000000,3.0,3.0,0,...,0,0.0,joff+,0.000000,2.666667,0.304348,3,0.600000,0.0,False
18,joff+_18,@ia_ronin Iyargel pero igusto avei.🤣,6,"{'url': {'has_duplicate': False, 'docs_ids': []}}",36,main_off_lang_dev.txt,0.000000,2.0,2.0,0,...,0,0.0,joff+,0.000000,5.250000,0.209677,0,0.571429,0.0,False
22,joff+_22,Nde tavy anocheeeeeeee🥴🥴🥴,6,"{'url': {'has_duplicate': False, 'docs_ids': []}}",25,main_off_lang_dev.txt,0.000000,3.0,3.0,0,...,0,0.0,joff+,0.500000,6.666667,0.478261,3,0.500000,0.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
875456,ancora_14115,"Si eso se considera plagio, yo te ñuaitĩ los q...",11,"{'url': {'has_duplicate': False, 'docs_ids': []}}",57,raw_ancora.tsv,0.000000,11.0,11.0,0,...,0,0.0,ancora,0.375000,4.090909,0.553191,11,0.846154,0.0,False
875459,ancora_14118,"Ha, ya estudiope, añoñakuri un punteo de mbara...",15,"{'url': {'has_duplicate': False, 'docs_ids': []}}",106,raw_ancora.tsv,0.066667,15.0,15.0,0,...,0,0.0,ancora,0.363636,5.800000,0.695652,15,0.789474,0.0,False
959860,joemo_134,@micaburgosbo @JRolon_19 @elClubOlimpia Ndetar...,6,"{'url': {'has_duplicate': False, 'docs_ids': []}}",71,main_emotion_test.txt,0.000000,3.0,3.0,0,...,0,2.0,joemo,0.000000,9.666667,0.590909,3,0.500000,0.0,False
959862,joemo_170,@riveros_vivi18 Heuse tova jepete ?,4,"{'url': {'has_duplicate': False, 'docs_ids': []}}",35,main_emotion_test.txt,0.000000,3.0,3.0,0,...,0,2.0,joemo,0.000000,5.000000,0.451613,3,0.600000,0.0,False


In [90]:
df_all[(df_all['num_words_punct_spacy'] < 2) & (df_all['language_score'] < 0.1)][['text', 'language_score']]

,text,language_score
296,tavycho,0.0
380,rehe,0.0
516,ivai,0.0
675,vyro,0.0
1304,heko,0.0
...,...,...
859809,chemongueraipa,0.0
860751,napepehína,0.0
861035,Arareko,0.0
861180,Oguah,0.0


In [ ]:
df_all[df_all['language_score']==0.0].fillna("None")["corpus"].value_counts()

corpus
grammar            4801
americasnlp2024    2947
ancora             2050
josa                417
joff+               307
gua_spa             259
commonvoice         171
tatoeba             170
bible               111
fineweb-2           106
madlad_clean         74
jojajovai            50
jofun                21
opus                 15
americasnlp2022      10
joemo                 3
belele                2
Name: count, dtype: int64

In [15]:
df_all["language_score_source"].fillna('None').value_counts()

language_score_source
glotlid        887785
openlid         51149
None            11514
strict-3of3      9705
Name: count, dtype: int64

In [93]:
df_all[df_all['ratio_symbols_to_words'] > 5]

,id,text,num_words_no_punct_spacy,duplicate,num_chars,corpus_file,avg_word_repetition_ratio_per_sentence,avg_words_per_sentence,avg_sentence_length,count_lorem_ipsum_sentences,...,count_sentences_with_javascript,avg_numbers_per_sentence,corpus,ratio_stopwords_to_non_stopwords,mean_word_length,avg_character_repetition_ratio_per_sentence,min_sentence_length,proportion_sentences_with_low_guarani_score,language_coefficient,remove_js
10,joff+_10,reko ☹️☹️☹️,7,"{'url': {'has_duplicate': False, 'docs_ids': []}}",11,main_off_lang_dev.txt,0.000000,1.000000,1.000000,0,...,0,0.000000,joff+,0.000000,4.000000,0.400000,1,0.142857,0.000000,False
84,joff+_84,@AngyFranco18 Añe'enamora jeyma😍😍😍😍😍,8,"{'url': {'has_duplicate': False, 'docs_ids': []}}",36,main_off_lang_dev.txt,0.000000,1.000000,1.000000,0,...,0,2.000000,joff+,0.000000,5.000000,0.470588,1,0.125000,7.142170,False
92,joff+_92,@ArzaVictor @fabian_1932 @joseayalaok Iporaite...,7,"{'url': {'has_duplicate': False, 'docs_ids': []}}",76,main_off_lang_dev.txt,0.000000,2.000000,2.000000,0,...,0,4.000000,joff+,0.000000,7.500000,0.571429,2,0.000000,inf,False
94,joff+_94,@Guarani__ nde reñe'e avañe'e?,4,"{'url': {'has_duplicate': False, 'docs_ids': []}}",30,main_off_lang_dev.txt,0.000000,1.000000,1.000000,0,...,0,0.000000,joff+,0.000000,3.000000,0.481481,1,0.142857,6.943681,False
114,joff+_114,@GokusenSayayin #poetadelgolvolve 😆😆😆😆 japuka ...,8,"{'url': {'has_duplicate': False, 'docs_ids': []}}",51,main_off_lang_dev.txt,0.000000,1.000000,1.000000,0,...,0,0.000000,joff+,0.000000,6.000000,0.489362,1,0.111111,5.745786,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
938523,Alpaca-gn-gpt4_34771,Emoambue ko ñe'ẽjoaju reipe'a'ỹre mba'eveichag...,13,"{'url': {'has_duplicate': False, 'docs_ids': []}}",124,data_train-00000-of-00001-0eb988ff2285b7a5.csv,0.000000,1.500000,1.500000,0,...,0,0.000000,Alpaca-gn-gpt4,0.000000,4.333333,0.568699,1,0.075000,13.093266,False
941971,Alpaca-gn-gpt4_38370,Ojejapo petet cuadro oñembojoja hagua mba éic...,109,"{'url': {'has_duplicate': False, 'docs_ids': []}}",714,data_train-00000-of-00001-0eb988ff2285b7a5.csv,0.489796,49.000000,49.000000,0,...,0,33.000000,Alpaca-gn-gpt4,0.020833,6.265306,0.914050,49,0.000000,inf,False
955093,twitter_giossa_october_337,@lariporro @juru_guasu Kuimba'e aty donde esta...,25,"{'url': {'has_duplicate': False, 'docs_ids': []}}",73,tweets_gn_october.csv,0.000000,3.000000,3.000000,0,...,0,0.000000,twitter_giossa_october,0.000000,5.000000,0.597015,3,0.000000,inf,False
956220,twitter_giossa_october_1464,@Apostalapy #VamosParaguay nde añarakopeguare ...,16,"{'url': {'has_duplicate': False, 'docs_ids': []}}",63,tweets_gn_october.csv,0.000000,2.000000,2.000000,0,...,0,0.000000,twitter_giossa_october,1.000000,8.500000,0.574074,2,0.000000,inf,False


In [92]:
df_all[(df_all['proportion_sentences_with_low_guarani_score']> 0.6) & (df_all['language_score'] != 0)]

,id,text,num_words_no_punct_spacy,duplicate,num_chars,corpus_file,avg_word_repetition_ratio_per_sentence,avg_words_per_sentence,avg_sentence_length,count_lorem_ipsum_sentences,...,count_sentences_with_javascript,avg_numbers_per_sentence,corpus,ratio_stopwords_to_non_stopwords,mean_word_length,avg_character_repetition_ratio_per_sentence,min_sentence_length,proportion_sentences_with_low_guarani_score,language_coefficient,remove_js
1,joff+_1,Che membyséma avei 🥰,4,"{'url': {'has_duplicate': False, 'docs_ids': []}}",20,main_off_lang_dev.txt,0.000000,3.0,3.0,0,...,0,0.000000,joff+,0.000000,5.333333,0.294118,3,0.750000,1.332920,False
2,joff+_2,Apostala mante ovy'a hina,4,"{'url': {'has_duplicate': False, 'docs_ids': []}}",25,main_off_lang_dev.txt,0.000000,3.0,3.0,0,...,0,0.000000,joff+,0.500000,5.666667,0.318182,3,0.750000,1.240805,False
5,joff+_5,Maitei peeme guarã,3,"{'url': {'has_duplicate': False, 'docs_ids': []}}",18,main_off_lang_dev.txt,0.000000,3.0,3.0,0,...,0,0.000000,joff+,0.500000,5.333333,0.312500,3,1.000000,0.851744,False
7,joff+_7,Napende kuerai piko la esquema preferidogui🙄,7,"{'url': {'has_duplicate': False, 'docs_ids': []}}",44,main_off_lang_dev.txt,0.000000,6.0,6.0,0,...,0,0.000000,joff+,0.500000,6.333333,0.538462,6,0.857143,1.004277,False
9,joff+_9,Ipochyma @SalvadorHicar “para eso nomás piko”,6,"{'url': {'has_duplicate': False, 'docs_ids': []}}",45,main_off_lang_dev.txt,0.000000,5.0,5.0,0,...,0,0.000000,joff+,0.666667,4.800000,0.425000,5,0.625000,1.031948,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
959880,joemo_1055,Mbaeteko los guiso kuera,4,"{'url': {'has_duplicate': False, 'docs_ids': []}}",24,main_emotion_train.txt,0.000000,4.0,4.0,0,...,0,0.000000,joemo,1.000000,5.250000,0.380952,4,1.000000,0.999722,False
959881,joemo_1114,ndovalei caca chera'a paraguay,4,"{'url': {'has_duplicate': False, 'docs_ids': []}}",30,main_emotion_train.txt,0.000000,3.0,3.0,0,...,0,0.000000,joemo,0.000000,6.666667,0.407407,3,0.750000,1.194365,False
959886,joemo_1484,Furro ndaha’éi gente (?),3,"{'url': {'has_duplicate': False, 'docs_ids': []}}",24,main_emotion_train.txt,0.000000,4.0,4.0,0,...,0,0.000000,joemo,0.000000,4.250000,0.190476,4,0.666667,1.485382,False
959890,joemo_1518,@juru_guasu @Chope55378982 Hola yuru mbaetecop...,6,"{'url': {'has_duplicate': False, 'docs_ids': []}}",52,main_emotion_train.txt,0.000000,4.0,4.0,0,...,0,8.000000,joemo,0.333333,5.500000,0.404255,4,0.666667,1.472568,False


In [10]:
bar_chart = px.bar(df_all, x='language_score_source')

In [ ]:
bar_chart.show()

In [ ]:
box_plot = px.box(df_all, y="num_words_punct_spacy", points="outliers", hover_data=["id"])
#box_plot.write_image(str(plots_dir / f"{metric}_box.png"))

In [ ]:
box_plot.show()

In [ ]:
diction = {'mean_word_length': 1313005,
 'count_sentences_with_low_guarani_proportion': 1313005,
 'avg_sentence_length': 1313005,
 'avg_uppercase_letters_per_sentence': 1313005,
 'count_sentences_ending_with_ellipsis': 1313005,
 'average_words_in_sentences_starting_with_capital': 1313005,
 'language_score': 1313005,
 'avg_word_repetition_ratio_per_sentence': 1313005,
 'count_sentences_starting_with_bullet': 1313005,
 'count_sentences_with_legal_phrases': 1313005,
 'avg_numbers_per_sentence': 1313005,
 'avg_character_repetition_ratio_per_sentence': 1313005,
 'num_words_no_punct_spacy': 1313005,
 'ratio_symbols_to_words': 1313005,
 'count_lorem_ipsum_sentences': 1313005,
 'num_words_punct_spacy': 1313005,
 'avg_alphanumeric_characters_per_sentence': 1313005,
 'num_words_split': 1313005,
 'avg_words_per_sentence': 1313005,
 'min_sentence_length': 1313005,
 'count_sentences_with_curly_bracket': 1313005,
 'count_sentences_with_javascript': 1313005,
 'ratio_stopwords_to_non_stopwords': 1313005,
 'max_sentence_length': 1313005,
 'count_bad_words_occurrences': 1313005,
 'avg_characters_per_sentence': 1313005,
 'count_sentences_without_terminal_punctuation': 1313005,
 'num_chars': 1313005,
}

In [ ]:
documents = iterate_directory(DATA_PATH)

In [ ]:
lists_to_explore = ["count_sentences_with_low_guarani_proportion", "count_sentences_with_javascript", "language_score", "avg_alphanumeric_characters_per_sentence", "average_words_in_sentences_starting_with_capital", "count_sentences_with_legal_phrases", "count_sentences_with_javascript"]

In [ ]:
rows = []
for doc in documents:
    row = {
        "doc_id":doc.id,
        "text": doc.text
    }

    for field in lists_to_explore:
        row[field] = doc.metadata[field]
    
    rows.append(row)

In [ ]:
plots_dir = PROJECT_ROOT/"outputs"/"heuristics_filtering"/"plots"
lists_dir = PROJECT_ROOT/"outputs"/"heuristics_filtering"/"lists"

In [ ]:
summary = df.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T
summary["n_unique"] = df.nunique()

In [ ]:
summary

In [ ]:
import plotly.express as px

In [ ]:
import plotly.io as pio
pio.renderers.default = "vscode"

In [ ]:
li_fig = px.box(df, y="count_lorem_ipsum_sentences", points="outliers", hover_data=["doc_id"])
li_fig.show()

In [ ]:
lp_fig = px.box(df, y="count_sentences_with_legal_phrases", points="outliers", hover_data=["doc_id"])
lp_fig.show()

In [ ]:
lp_fig.write_html(str(PROJECT_ROOT/"outputs"/"test.html"))

In [94]:
def find_outliers_iqr(df, field):
    q1 = df[field].quantile(0.25)
    q3 = df[field].quantile(0.75)

    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = df[
        (df[field] < lower) |
        (df[field] > upper)
    ]

    return outliers

In [ ]:
heu_fil = PROJECT_ROOT/"outputs"/"heuristics_filtering"

In [ ]:
df.head()

In [ ]:
outliers_df = find_outliers_iqr(df_all, 'ratio_symbols_to_words')

In [ ]:
outliers_df.to_json(str(PROJECT_ROOT/ 'outputs'/ 'heuristics_filtering' /'symbols_to_words.jsonl'), orient='records', lines=True, force_ascii=False)

: 

In [ ]:
outliers_df

In [ ]:
for metric in lists_to_explore:
    outliers_df = find_outliers_iqr(df, metric)
    outliers_dict = outliers_df.to_dict(orient="records")

    with open(str(heu_fil / f"{metric}_texts.jsonl"), "w") as file:
        for entry in outliers_dict:
            json_record = json.dumps(entry, ensure_ascii=False)
            file.write(json_record + '\n')

In [ ]:
for metric in list(diction.keys()):
    #Generate a box plot
    box_plot = px.box(df, y=metric, points="outliers", hover_data=["doc_id"])
    box_plot.write_image(str(plots_dir / f"{metric}_box.png"))

    #Get the outliers
    outliers_df = find_outliers_iqr(df, metric)
    outliers_list = outliers_df["doc_id"].tolist()

    with open(str(lists_dir / f"{metric}.txt"), "w") as file:
        file.write("\n".join(outliers_list))

    #Generate a histogram
    histogram = px.histogram(df, x=metric, hover_data=["doc_id"], marginal="box")
    histogram.write_image(str(plots_dir / f"{metric}_histogram.png"))

In [ ]:
import kaleido

kaleido.get_chrome_sync()

In [ ]:
# fig = px.histogram(
#     df,
#     x="mean_word_length",
#     hover_data=["doc_id"]
# )

In [ ]:
# fig.show()

In [ ]:
df.dtypes

In [ ]:
documents

In [ ]:
w_js = [
    "opus-all-en_363609", 
    "opus-all-en_363610", 
    "opus-all-en_363611", 
    "opus-all-en_363612", 
    "opus-all-en_381869", 
    "americasnlp2024_27377",
    "americasnlp2024_27378", 
    "fineweb-2_365", 
    "fineweb-2_748", 
    "fineweb-2_1651", 
    "fineweb-2_3042", 
    "fineweb-2_3042",  
    "fineweb-2_4016", 
    "fineweb-2_4500",
    "fineweb-2_7468",
    "fineweb-2_11951",
    "fineweb-2_12334"
    "fineweb-2_13237",
    "fineweb-2_23920",
    "fineweb-2_24823",
    "fineweb-2_26214",
    "fineweb-2_27188", 
    "fineweb-2_27672", 
    "fineweb-2_30640", 
    "fineweb-2_36209", 
    "fineweb-2_37078", 
    "fineweb-2_37116", 
    "fineweb-2_37525", 
    "fineweb-2_37527", 
    "fineweb-2_37952", 
    "fineweb-2_38176", 
    "fineweb-2_38881", 
    "fineweb-2_41411", 
    "fineweb-2_41689", 
    "fineweb-2_41856", 
    "fineweb-2_43121", 
    "fineweb-2_43954", 
    "fineweb-2_44035", 
    "fineweb-2_44199", 
    "fineweb-2_44309", 
    "fineweb-2_44993"
]

In [ ]:
len(w_js)

In [ ]:
metrics = ["count_sentences_with_low_guarani_proportion", "count_sentences_with_javascript", "language_score", "language", "num_chars"]

In [ ]:
exp_rows = []
for doc in documents:
    if doc.id in w_js:
        row = {
            "doc_id":doc.id,
            "text": doc.text
        }

        for field in metrics:
            row[field] = doc.metadata[field]
        
        exp_rows.append(row)

In [ ]:
exp_df = pd.DataFrame(exp_rows)

In [ ]:
import pandas as pd
import plotly.express as px

# Keep only needed columns and drop missing values
plot_df = exp_df.dropna()

fig = px.scatter(
    plot_df,
    x="language_score",
    y="count_sentences_with_low_guarani_proportion",
    #size="language_score",          # dot size
    color="language",              # dot colour
    hover_data=["doc_id", "num_chars", "language_score", "count_sentences_with_javascript"],
    size_max=40,
    opacity=0.7,
    title="Low Guarani Proportion vs Javascript Sentences"
)

# fig.update_layout(
#     xaxis_title="Count Sentences with Low Guarani Proportion",
#     yaxis_title="Count Sentences with Javascript",
#     template="plotly_white"
# )

fig.show()